# C01 — native land: cessions (Royce), state cessions, pre-1784 lines, reserves, removal routes
Task `tasks/T-007-c01-native-layers.md` (brief B-018). Layers 2 and 5 (`docs/PLAN.md` §2). File reads + joins only; no ML/LLM.

**Sources**
| Input | What | Grain | Licence |
|---|---|---|---|
| `data/raw/gis/usfs-tribal-ceded-lands/shp/` + `table/…gdb` | USFS Tribal Ceded Lands = Royce (1899) cession areas, EPSG:4269 | one row per Royce cession number (`CESSNUM`) | CC BY 4.0 |
| `research/treaties_R1.csv` | R1 treaty list (a selection) | treaty | ours |
| `data/processed/reservations_points.csv` | R4 reserves + T-002 points | reserve | ours (coords: OSM/ODbL, Newberry, Royce) |
| `data/raw/gis/nps-trail-of-tears/` | NPS Trail of Tears NHT designated route, 3 GeoJSON pages | line segment | US federal work |
| `data/raw/gis/newberry-ahcb-counties/` | modern county polygons (`END_DATE` 2000), only to place the NY state-cession points | county | free, cite |

**Outputs** (`web/data/`): `native_land.topojson` (objects `royce_cessions`, `state_cessions`, `pre1784_lines`), `reservations.json`, `removal_routes.topojson` (object `trail_of_tears`).

**Caveats** (R7, `docs/DATA.md`)
- Royce maps the United States' acquisitions, drawn by Royce's staff in the 1890s from treaty wording; it is not a map of where nations lived. Nation names are as the cession schedule writes them (`SCHDTRB`), often decades after 1784.
- Royce has no NY state cessions 1785–89 and no New England areas; those appear here as points (`source = not_royce`) or not at all.
- Pre-1784 lines have no open vector source (R6): placeholders only, `todo = true`.
- The Trail of Tears is one NPS designated route, undated, digitised at 1:100,000; it does not say which nation or which year.

In [1]:
import sys, json, re
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path: sys.path.insert(0, str(PIPELINE_DIR))
import config
import duckdb, pandas as pd, numpy as np, geopandas as gpd, pyogrio, shapely
from shapely.geometry import shape, mapping
pd.set_option('display.width',220); pd.set_option('display.max_rows',120); pd.set_option('display.max_colwidth',90); pd.set_option('display.max_columns',40)
con = duckdb.connect()
def q(sql): return con.sql(sql).df()

GIS = config.RAW_DIR / 'gis'
ROYCE_SHP = GIS / 'usfs-tribal-ceded-lands/shp/S_USA.TRIBALCEDEDLANDS.shp'
ROYCE_TBL = GIS / 'usfs-tribal-ceded-lands/table/S_USA.TribalCededLandsTable.gdb'
COUNTIES  = GIS / 'newberry-ahcb-counties/shp/US_HistCounties_Gen001.shp'
TRAIL     = sorted((GIS / 'nps-trail-of-tears').glob('TRTE_NHT_off*.geojson'))
R1_CSV    = config.PROJECT_DIR / 'research/treaties_R1.csv'
RES_CSV   = config.PROCESSED_DIR / 'reservations_points.csv'
WEB_DIR   = config.PROJECT_DIR / 'web/data'
WEB_DIR.mkdir(parents=True, exist_ok=True)
WINDOW_END = 9999          # no cutoff: every dated Royce row. L2 = land not yet ceded at the slider date, so areas ceded after 1840 must be drawn (Lara, 2026-11-23)
QUANT = 100_000            # TopoJSON quantization (about 50 m at this extent)

DATASET = {
    'royce': 'USDA Forest Service, Tribal Ceded Lands (Royce 1899), CC BY 4.0',
    'r1':    'R1 treaties (research/treaties_R1.csv)',
    'r4':    'R4 reservations + T-002 points (data/processed/reservations_points.csv)',
    'nps':   'National Park Service 2021, Trail of Tears NHT designated route',
}
print('duckdb', duckdb.__version__, '| geopandas', gpd.__version__, '| shapely', shapely.__version__)

duckdb 2.0.0 | geopandas 1.2.0 | shapely 2.2.0


## 0. TopoJSON writer
No TopoJSON package is installed and the task allows no new dependency, so a small encoder: one arc per ring or line (no shared arcs),
quantized and delta-encoded per the TopoJSON spec. Null geometries (`"type": null`) carry rows with no shape. The decoder is used only to check the round trip in §6.

In [2]:
def _clean(v):
    """JSON-safe scalar: NaN/NaT -> None, numpy -> python, timestamps -> ISO date."""
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if v is pd.NaT: return None
    if isinstance(v, pd.Timestamp): return v.strftime('%Y-%m-%d')
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return None if np.isnan(v) else float(v)
    if isinstance(v, np.bool_): return bool(v)
    try:
        if pd.isna(v): return None
    except (TypeError, ValueError): pass
    return v

def clean_props(d): return {k: _clean(v) for k, v in d.items()}

def topo_encode(objects, n=QUANT):
    """objects: {name: [(shapely geom or None, props dict, id), ...]} -> TopoJSON dict."""
    geoms = [g for feats in objects.values() for g, _, _ in feats if g is not None]
    x0, y0, x1, y1 = shapely.total_bounds(np.array(geoms, dtype=object))
    kx, ky = (x1 - x0) / (n - 1), (y1 - y0) / (n - 1)
    arcs = []
    def qpt(x, y): return [int(round((x - x0) / kx)), int(round((y - y0) / ky))]
    def arc(coords, ring=False):
        pts = [qpt(x, y) for x, y, *_ in coords]
        dd = [pts[0]] + [p for a, p in zip(pts, pts[1:]) if p != a]
        if len(dd) >= (4 if ring else 2): pts = dd        # keep tiny rings un-deduplicated so none vanish
        delta = [pts[0]] + [[b[0] - a[0], b[1] - a[1]] for a, b in zip(pts, pts[1:])]
        arcs.append(delta); return len(arcs) - 1
    def poly(p): return [arc(p.exterior.coords, True)] + [arc(r.coords, True) for r in p.interiors]
    def enc(g):
        if g is None: return {'type': None}
        t = g.geom_type
        if t == 'Polygon':         return {'type': t, 'arcs': poly(g)}
        if t == 'MultiPolygon':    return {'type': t, 'arcs': [poly(p) for p in g.geoms]}
        if t == 'LineString':      return {'type': t, 'arcs': [arc(g.coords)]}
        if t == 'MultiLineString': return {'type': t, 'arcs': [arc(l.coords) for l in g.geoms]}
        if t == 'Point':           return {'type': t, 'coordinates': qpt(g.x, g.y)}
        raise ValueError(t)
    out = {'type': 'Topology', 'bbox': [x0, y0, x1, y1],
           'transform': {'scale': [kx, ky], 'translate': [x0, y0]}, 'objects': {}, 'arcs': arcs}
    for name, feats in objects.items():
        gs = []
        for g, props, fid in feats:
            e = enc(g); e['properties'] = clean_props(props)
            if fid is not None: e['id'] = fid
            gs.append(e)
        out['objects'][name] = {'type': 'GeometryCollection', 'geometries': gs}
    return out

def topo_decode(topo, name):
    """-> list of (shapely geom or None, props). Inverse of topo_encode (no shared arcs)."""
    (kx, ky), (tx, ty) = topo['transform']['scale'], topo['transform']['translate']
    def arc(i):
        x = y = 0; pts = []
        for dx, dy in topo['arcs'][i]:
            x += dx; y += dy; pts.append((x * kx + tx, y * ky + ty))
        return pts
    def dec(g):
        t = g['type']
        if t is None: return None
        if t == 'Point': x, y = g['coordinates']; return shapely.Point(x * kx + tx, y * ky + ty)
        if t == 'LineString': return shapely.LineString(arc(g['arcs'][0]))
        if t == 'MultiLineString': return shapely.MultiLineString([arc(a) for a in g['arcs']])
        if t == 'Polygon': return shapely.Polygon(arc(g['arcs'][0]), [arc(a) for a in g['arcs'][1:]])
        if t == 'MultiPolygon': return shapely.MultiPolygon([(arc(p[0]), [arc(a) for a in p[1:]]) for p in g['arcs']])
        raise ValueError(t)
    return [(dec(g), g.get('properties', {})) for g in topo['objects'][name]['geometries']]

def write_json(obj, path):
    s = json.dumps(obj, ensure_ascii=False, allow_nan=False, separators=(',', ':'))
    path.write_text(s, encoding='utf-8'); print(f'wrote {path.relative_to(config.PROJECT_DIR)}  {len(s)/1e6:.2f} MB')

# self-test on a square with a hole, a line and a point
_t = topo_encode({'t': [(shapely.Polygon([(0,0),(0,1),(1,1),(1,0)], [[(.2,.2),(.4,.2),(.4,.4),(.2,.4)]]), {'a': np.nan}, 'x'),
                        (shapely.MultiLineString([[(0,0),(1,1)], [(1,0),(0,1)]]), {}, None),
                        (shapely.Point(.5,.5), {}, None), (None, {'b': 1}, None)]})
_d = topo_decode(_t, 't')
assert _d[0][0].equals_exact(shapely.Polygon([(0,0),(0,1),(1,1),(1,0)], [[(.2,.2),(.4,.2),(.4,.4),(.2,.4)]]), 1e-4)
assert _d[1][0].geom_type == 'MultiLineString' and len(_d[1][0].geoms) == 2 and _d[3][0] is None and _d[0][1]['a'] is None
print('topojson self-test ok')

topojson self-test ok


## 1. Royce cession areas (USFS Tribal Ceded Lands)
The shapefile holds geometry + `CESSNUM`; the table holds the dates (`CESSDATE1..5`), the nations as the schedule names them (`SCHDTRB`),
present-day nations, citations and links. Join on `CESSNUM`. The **table drives**: a dated Royce row with no polygon is kept with a null geometry and `flag = no_geometry`.

In [3]:
shp = gpd.read_file(ROYCE_SHP)
tbl = pyogrio.read_dataframe(ROYCE_TBL)
DATE_COLS = [f'CESSDATE{i}' for i in range(1, 6)]
tbl['_earliest'] = tbl[DATE_COLS].min(axis=1)
print('shapefile', len(shp), shp.crs, '| table', len(tbl))
print('null CESSNUM: shp', shp.CESSNUM.isna().sum(), '| table', tbl.CESSNUM.isna().sum(), '| duplicate CESSNUM: shp', shp.CESSNUM.dropna().duplicated().sum(), '| table', tbl.CESSNUM.dropna().duplicated().sum())
print('table rows with no date:', tbl._earliest.isna().sum(), '| earliest', tbl._earliest.min().date(), '| latest', tbl._earliest.max().date())
print('table CESSNUMs with no polygon:', sorted(set(tbl.CESSNUM.dropna()) - set(shp.CESSNUM.dropna())))
print('polygons with no table row:', sorted(set(shp.CESSNUM.dropna()) - set(tbl.CESSNUM.dropna())), '| shp row with null CESSNUM bounds:', shp[shp.CESSNUM.isna()].total_bounds.round(1))
print('dates by stored hour (UTC):', tbl[DATE_COLS].stack().dt.hour.value_counts().to_dict(), '-> read as calendar dates (see 1a for a one-day offset)')
print('rows by number of dates:', tbl[DATE_COLS].notna().sum(axis=1).value_counts().sort_index().to_dict())

shapefile 718 EPSG:4269 | table 725
null CESSNUM: shp 1 | table 1 | duplicate CESSNUM: shp 0 | table 0
table rows with no date: 1 | earliest 1784-10-21 | latest 1893-02-19
table CESSNUMs with no polygon: ['128', '129', '130', '47', '543', '65', '690']
polygons with no table row: [] | shp row with null CESSNUM bounds: [-110.9   32.5 -109.1   33.4]
dates by stored hour (UTC): {12.0: 1166, 0.0: 4} -> read as calendar dates (see 1a for a one-day offset)
rows by number of dates: {0: 1, 1: 405, 2: 221, 3: 72, 4: 23, 5: 3}


In [4]:
# Scope: every dated Royce row (areas ceded after 1840 stay 'not yet ceded' for the whole 1784–1840 slider)
roy = tbl[tbl._earliest.dt.year <= WINDOW_END].copy()
roy = roy.merge(shp.to_crs(4326)[['CESSNUM', 'CESSNUMSOR', 'SHAPE_AREA', 'SHAPE_LEN', 'geometry']], on='CESSNUM', how='left', validate='1:1')
roy['cessnum'] = roy.CESSNUM
roy['nation_as_named'] = roy.SCHDTRB
roy['cession_date'] = roy._earliest.dt.strftime('%Y-%m-%d')
roy['cession_year'] = roy._earliest.dt.year.astype(int)
roy['cession_dates'] = roy[DATE_COLS].apply(lambda r: ';'.join(sorted(d.strftime('%Y-%m-%d') for d in r.dropna())), axis=1)
roy['n_dates'] = roy[DATE_COLS].notna().sum(axis=1)
for c in DATE_COLS: roy[c] = roy[c].dt.strftime('%Y-%m-%d')
roy['has_geometry'] = roy.geometry.notna()
roy['flag'] = np.where(roy.has_geometry, None, 'no_geometry')
roy['kind'] = 'cession_polygon'; roy['source'] = 'royce'; roy['dataset'] = DATASET['royce']
roy['year'] = roy.cession_year; roy['year_flag'] = None; roy['todo'] = False

# R1 labels: R1 rows that list this Royce number
r1 = pd.read_csv(R1_CSV, dtype=str)
r1x = r1.assign(cessnum=r1.royce_cession_numbers.str.split(';')).explode('cessnum').dropna(subset=['cessnum'])
r1x['cessnum'] = r1x.cessnum.str.strip()
lab = r1x.groupby('cessnum').agg(r1_treaty_id=('id', ';'.join), r1_treaty_name=('name', ';'.join), r1_date_signed=('date_signed', ';'.join)).reset_index()
roy = roy.merge(lab, on='cessnum', how='left', validate='1:1').drop(columns='_earliest')

print('Royce rows (all dated):', len(roy), '| with polygon:', roy.has_geometry.sum(), '| no polygon:', (~roy.has_geometry).sum())
print('invalid polygons in scope:', (~roy[roy.has_geometry].geometry.apply(lambda g: g.is_valid)).sum())
print('with an R1 label:', roy.r1_treaty_id.notna().sum(), '| R1 Royce numbers not in scope:', sorted(set(r1x.cessnum) - set(roy.cessnum)))
roy.loc[~roy.has_geometry, ['cessnum', 'MAPNAME', 'SCHDTRB', 'cession_date', 'STATE']]
print('ceded after 1840:', (roy.cession_year > 1840).sum(), '| earliest date on 31 Dec:', (roy.cession_date.str.endswith('-12-31')).sum())


Royce rows (all dated): 724 | with polygon: 717 | no polygon: 7


invalid polygons in scope: 3
with an R1 label: 65 | R1 Royce numbers not in scope: []
ceded after 1840: 425 | earliest date on 31 Dec: 2


### 1a. Overlaps and R1 date check
Overlapping cessions stay separate polygons (one per Royce number); nothing is dissolved. R1 dates are compared with Royce's earliest date for the same number.
This is a check only: the USFS dates are written to the files as stored. Where R1's treaty is the polygon's first cession, the USFS date is one day earlier than R1 (Kappler) in every case;
the four dates stored at 00:00 UTC are not shifted. Years and decades are not affected unless an earliest date falls on 31 December (checked in the print below). The rows below with large gaps are polygons R1 cites for a later, second cession.

In [5]:
g = gpd.GeoDataFrame(roy[roy.has_geometry][['cessnum', 'geometry']], crs=4326).to_crs(5070)
g['geometry'] = g.geometry.make_valid()
pairs = gpd.sjoin(g, g, predicate='intersects')
pairs = pairs[pairs.cessnum_left < pairs.cessnum_right]
right = g.set_index('cessnum').geometry
pairs['shared_km2'] = [a.intersection(right[b]).area / 1e6 for a, b in zip(pairs.geometry, pairs.cessnum_right)]
big = pairs[pairs.shared_km2 > 1]
print('polygon pairs sharing > 1 km2:', len(big), '| polygons involved:', len(set(big.cessnum_left) | set(big.cessnum_right)))

# R1 date_signed vs Royce earliest date, per (R1 treaty, Royce number)
chk = r1x.merge(roy[['cessnum', 'cession_date', 'cession_dates']], on='cessnum')
chk['days_r1_minus_royce'] = (pd.to_datetime(chk.date_signed) - pd.to_datetime(chk.cession_date)).dt.days
print('days R1 minus Royce earliest:', chk.days_r1_minus_royce.value_counts().sort_index().to_dict())
print('Royce dates stored at 00:00 UTC (the rest are 12:00):')
display(tbl[DATE_COLS].stack().loc[lambda s: s.dt.hour == 0].rename('date').reset_index().merge(tbl[['CESSNUM']], left_on='level_0', right_index=True)[['CESSNUM', 'level_1', 'date']])
print('in-scope Royce earliest dates on 31 Dec (a one-day shift would change the year):', (roy.cession_date.str.endswith('-12-31')).sum())
chk[chk.days_r1_minus_royce != 1][['id', 'cessnum', 'date_signed', 'cession_date', 'cession_dates', 'days_r1_minus_royce']]

polygon pairs sharing > 1 km2: 706 | polygons involved: 600
days R1 minus Royce earliest: {1: 56, 335: 1, 2200: 1, 3157: 1, 4412: 1, 4874: 4, 9082: 1}
Royce dates stored at 00:00 UTC (the rest are 12:00):


,CESSNUM,level_1,date
0,272,CESSDATE2,1856-02-05 00:00:00+00:00
1,128,CESSDATE2,1826-01-24 00:00:00+00:00
2,129,CESSDATE2,1826-01-24 00:00:00+00:00
3,130,CESSDATE2,1826-01-24 00:00:00+00:00


in-scope Royce earliest dates on 31 Dec (a one-day shift would change the year): 2


,id,cessnum,date_signed,cession_date,cession_dates,days_r1_minus_royce
47,T1819,105,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
49,T1819,107,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
50,T1819,108,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
51,T1819,109,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
56,T1830,157,1830-09-27,1805-11-15,1805-11-15;1830-09-26,9082
58,T1832S,173,1832-05-09,1823-09-17,1823-09-17;1832-05-08,3157
61,T1833,188,1833-09-26,1821-08-28,1821-08-28;1832-10-26;1833-09-26,4412
62,T1833,189,1833-09-26,1827-09-18,1827-09-18;1833-09-26,2200
63,T1833,190,1833-09-26,1832-10-26,1832-10-26;1833-09-26,335


In [6]:
con.register('roy_v', roy.drop(columns='geometry'))
q("""select (cession_year::int // 10) * 10 as decade, count(*) royce_rows, count(*) filter (where has_geometry) polygons,
     count(*) filter (where n_dates > 1) multi_date, count(*) filter (where r1_treaty_id is not null) with_r1
     from roy_v group by 1 order by 1""")

,decade,royce_rows,polygons,multi_date,with_r1
0,1780,6,6,3,3
1,1790,36,36,7,23
2,1800,55,52,33,17
3,1810,71,71,45,10
4,1820,47,45,23,6
5,1830,84,84,51,6
6,1840,18,18,10,0
7,1850,171,171,62,0
8,1860,67,67,30,0
9,1870,82,81,43,0


## 2. NY state and private cessions not in Royce (R1) → dated points
R1 rows with no Royce number, sold to New York State or to Phelps and Gorham, 1785–89. Point = centroid of the modern counties R1's `present_day_area` names
(Newberry, `END_DATE` 2000, same method as T-002). Two rows name no county: the marker county is an inference (`coord_inferred = true`). The points mark the area; they are not its extent.

In [7]:
STATE_CESSIONS = {   # R1 id -> modern NY counties used for the marker
    'T1785NY': (['BROOME'], True,  'R1 names only the "present Binghamton area" (Broome Co.); the cession was larger.'),
    'T1788PG': (['ONTARIO', 'YATES', 'STEUBEN', 'MONROE'], False, 'Counties as listed in R1 present_day_area.'),
    'T1788ON': (['ONONDAGA', 'CORTLAND', 'OSWEGO'], False, 'Counties as listed in R1 present_day_area.'),
    'T1788OE': (['ONEIDA'], True,  'R1 names no county ("Central New York; ... near Oneida Lake"); Oneida Co. is a marker only.'),
    'T1789CA': (['CAYUGA', 'SENECA'], False, 'Counties as listed in R1 present_day_area.'),
}
cty = gpd.read_file(COUNTIES)
cty = cty[(cty.STATE_TERR == 'New York') & (cty.END_DATE.dt.year == 2000)].to_crs(5070)
sc = r1[r1.id.isin(STATE_CESSIONS)].copy()
assert len(sc) == len(STATE_CESSIONS) and sc.royce_cession_numbers.isna().all()
geoms, counties, inferred, notes = [], [], [], []
for rid in sc.id:
    names, inf, note = STATE_CESSIONS[rid]
    sel = cty[cty.NAME.isin(names)]; assert len(sel) == len(names), (rid, names)
    geoms.append(gpd.GeoSeries([sel.union_all().centroid], crs=5070).to_crs(4326).iloc[0])
    counties.append(';'.join(names)); inferred.append(inf); notes.append(note)
sc['geometry'] = geoms
sc['kind'] = 'state_cession_point'; sc['source'] = 'not_royce'; sc['dataset'] = DATASET['r1']
sc['year'] = sc.date_signed.str[:4].astype(int)
sc['date_precision'] = np.where(sc.date_signed.str.len() == 10, 'day', np.where(sc.date_signed.str.len() == 7, 'month', 'year'))
sc['year_flag'] = None; sc['todo'] = False
sc['coord_method'] = 'county_centroid'; sc['coord_counties'] = counties; sc['coord_inferred'] = inferred; sc['coord_note'] = notes
sc['coord_source'] = 'Newberry AHCB county polygons (modern extent, END_DATE 2000), data/raw/gis/newberry-ahcb-counties'
sc['lat'] = sc.geometry.apply(lambda p: round(p.y, 4)); sc['lon'] = sc.geometry.apply(lambda p: round(p.x, 4))
print('other R1 rows with no Royce number (not drawn here):', r1[r1.royce_cession_numbers.isna() & ~r1.id.isin(STATE_CESSIONS)].id.tolist())
sc[['id', 'name', 'date_signed', 'year', 'date_precision', 'native_parties', 'coord_counties', 'coord_inferred', 'lat', 'lon']]

other R1 rows with no Royce number (not drawn here): ['P1763', 'T1768HL', 'T1768FS', 'T1770', 'T1774', 'T1775', 'T1777D', 'T1777L', 'T1783GC', 'T1783GK', 'T1784PA', 'T1785PA', 'T1785FM', 'L1785', 'T1786S', 'H1786', 'N1787', 'T1789FH', 'R1830']


,id,name,date_signed,year,date_precision,native_parties,coord_counties,coord_inferred,lat,lon
15,T1785NY,Treaty of Fort Herkimer,1785-06,1785,month,Oneida; Tuscarora,BROOME,True,42.1602,-75.8194
20,T1788PG,Phelps–Gorham Purchase (First Treaty of Buffalo Creek),1788-07-08,1788,day,Seneca (participation of other Haudenosaunee nations unverified),ONTARIO;YATES;STEUBEN;MONROE,False,42.6249,-77.3986
21,T1788ON,New York–Onondaga treaty,1788-09-12,1788,day,Onondaga,ONONDAGA;CORTLAND;OSWEGO,False,43.1007,-76.1445
22,T1788OE,New York–Oneida treaty,1788-09-22,1788,day,Oneida,ONEIDA,True,43.2415,-75.4359
24,T1789CA,New York–Cayuga treaty,1789-02-25,1789,day,Cayuga,CAYUGA;SENECA,False,42.8691,-76.6477


## 3. Pre-1784 lines → placeholders
Proclamation of 1763 and Fort Stanwix 1768: no open vector exists (R6); hand-tracing is a later task. Features have a null geometry, the R1 row, and `todo = true`.

In [8]:
PRE1784 = {'P1763': 'Proclamation Line of 1763 (South: De Vorsey 1966)', 'T1768FS': 'Fort Stanwix 1768 line (North: NPS description in R1)'}
pl = r1[r1.id.isin(PRE1784)].copy(); assert len(pl) == len(PRE1784)
pl['line_name'] = pl.id.map(PRE1784)
pl['kind'] = 'boundary_line'; pl['source'] = 'not_royce'; pl['dataset'] = DATASET['r1']
pl['year'] = pl.date_signed.str[:4].astype(int); pl['year_flag'] = None
pl['todo'] = True; pl['todo_note'] = 'Hand-trace the line (later task; no open vector source, R6).'; pl['flag'] = 'no_geometry'
pl[['id', 'line_name', 'date_signed', 'year', 'todo']]

,id,line_name,date_signed,year,todo
0,P1763,Proclamation Line of 1763 (South: De Vorsey 1966),1763-10-07,1763,True
2,T1768FS,Fort Stanwix 1768 line (North: NPS description in R1),1768-11-05,1768,True


In [9]:
def feats(df, idcol):
    cols = [c for c in df.columns if c != 'geometry']
    return [(None if (g is None or (isinstance(g, float) and np.isnan(g))) else g, dict(zip(cols, row)), fid)
            for g, row, fid in zip(df.geometry, df[cols].itertuples(index=False, name=None), df[idcol])]

native = topo_encode({
    'royce_cessions': feats(roy, 'cessnum'),
    'state_cessions': feats(sc, 'id'),
    'pre1784_lines':  feats(pl.assign(geometry=None), 'id'),
})
write_json(native, WEB_DIR / 'native_land.topojson')
print({k: len(v['geometries']) for k, v in native['objects'].items()}, '| arcs', len(native['arcs']))

wrote web/data/native_land.topojson  2.70 MB
{'royce_cessions': 724, 'state_cessions': 5, 'pre1784_lines': 2} | arcs 748


## 4. Reserves (R4 + T-002) → `reservations.json`
Every CSV column is kept. Added: `start_year`, `end_year` and their flags, parsed from the text dates. Rule: the first four-digit year;
flag `exact` for a plain date (`1797-09-15`, `1646`), `approx` for anything else (`c. 1705`, `1820s`, `1801–1802`, `1807 (or 1805)`).
No year when the text says the reserve continues, ended after 1840, or was only reduced; the flag says which. 7 rows have no point (`coord_method = none`, dispersed allotment classes or no place).

In [10]:
res = pd.read_csv(RES_CSV)
YEAR = re.compile(r'(1[5-9]\d\d)')
EXACT = re.compile(r'^\d{4}(-\d{2}(-\d{2})?)?$')
def parse(txt, end=False):
    if txt is None or (isinstance(txt, float) and np.isnan(txt)): return None, 'no_date'
    t = str(txt).strip(); low = t.lower()
    if end and 'continuing' in low: return None, 'continuing'
    if end and low.startswith('(after'): return None, 'after_1840'
    if end and 'reduced' in low: return None, 'reduced_only'
    m = YEAR.search(t)
    if not m: return None, 'unparsed'
    return int(m.group(1)), ('exact' if EXACT.match(t) else 'approx')
res[['start_year', 'start_flag']] = [parse(v) for v in res.date_established]
res[['end_year', 'end_flag']] = [parse(v, end=True) for v in res.date_ended]
res['start_year'] = res.start_year.astype('Int64'); res['end_year'] = res.end_year.astype('Int64')
res['has_coords'] = res.lat.notna() & res.lon.notna()
res['dataset'] = DATASET['r4']
res['year'] = res.start_year; res['year_flag'] = np.where(res.start_year.isna(), res.start_flag, None)
print('start flags', res.start_flag.value_counts().to_dict(), '| end flags', res.end_flag.value_counts().to_dict(), '| no coords', (~res.has_coords).sum())
res[(res.start_flag != 'exact') | (res.end_flag != 'exact')][['id', 'date_established', 'start_year', 'start_flag', 'date_ended', 'end_year', 'end_flag']]

start flags {'exact': 61, 'no_date': 3, 'approx': 2} | end flags {'exact': 26, 'no_date': 19, 'continuing': 11, 'approx': 5, 'after_1840': 3, 'reduced_only': 2} | no coords 7


,id,date_established,start_year,start_flag,date_ended,end_year,end_flag
0,R4-001,1646,1646,exact,NaN,<NA>,no_date
1,R4-002,NaN,<NA>,no_date,NaN,<NA>,no_date
3,R4-004,c. 1705,1705,approx,1824 (allotment begins); last parcels 1870s,1824,approx
5,R4-006,1758,1758,exact,1801–1802,1801,approx
6,R4-007,1788-09-22,1788,exact,1795–1846 (reduced),<NA>,reduced_only
7,R4-008,1788-09-12,1788,exact,(reduced 1793 and 1795),<NA>,reduced_only
8,R4-009,1789-02-25,1789,exact,1807 (or 1805),1807,approx
9,R4-010,mid-1780s,1780,approx,1820s,1820,approx
10,R4-011,1774-01-20,1774,exact,1820s–1830s,1820,approx
16,R4-017,1797-09-15,1797,exact,NaN,<NA>,no_date


In [11]:
write_json([clean_props(r) for r in res.to_dict('records')], WEB_DIR / 'reservations.json')
con.register('res_v', res)
q("""with s as (select (start_year::int // 10) * 10 as decade, count(*) started from res_v where start_year is not null group by 1),
          e as (select (end_year::int // 10) * 10 as decade, count(*) ended from res_v where end_year is not null group by 1)
     select decade, coalesce(started, 0) started, coalesce(ended, 0) ended from s full join e using (decade) order by decade""")

wrote web/data/reservations.json  0.09 MB


,decade,started,ended
0,1640,2,0
1,1680,1,0
2,1700,1,0
3,1730,1,0
4,1750,1,0
5,1760,1,0
6,1770,1,0
7,1780,4,0
8,1790,15,1
9,1800,1,2


## 5. Trail of Tears (NPS) → `removal_routes.topojson`
Three GeoJSON pages of one ArcGIS query (offsets 0, 1000, 2000), concatenated. All NPS columns kept. Undated: `year = null`, `year_flag = undated`.
`SOURCEDATE`, `CREATEDATE`, `EDITDATE` are NPS record dates in epoch ms (not event dates).

In [12]:
tt = pd.concat([gpd.read_file(p) for p in TRAIL], ignore_index=True)
tt = gpd.GeoDataFrame(tt, geometry='geometry', crs=4326)
print('pages', [p.name for p in TRAIL], '| features', len(tt), '| duplicate OBJECTID', tt.OBJECTID.duplicated().sum(), '| duplicate GlobalID', tt.GlobalID.duplicated().sum())
print('geometry', tt.geom_type.value_counts().to_dict(), '| empty', tt.is_empty.sum(), '| null', tt.geometry.isna().sum())
tt['kind'] = 'removal_route'; tt['source'] = 'nps_trte'; tt['dataset'] = DATASET['nps']
tt['year'] = None; tt['year_flag'] = 'undated'
display(tt.TRALTNAME.value_counts(dropna=False).to_frame('segments'))
display(tt.DATAACCESS.value_counts(dropna=False).to_frame('segments'))
routes = topo_encode({'trail_of_tears': feats(tt, 'OBJECTID')})
write_json(routes, WEB_DIR / 'removal_routes.topojson')

pages ['TRTE_NHT_off0.geojson', 'TRTE_NHT_off1000.geojson', 'TRTE_NHT_off2000.geojson'] | features 2006 | duplicate OBJECTID 0 | duplicate GlobalID 0
geometry {'LineString': 1381, 'MultiLineString': 625} | empty 0 | null 0


,segments
TRALTNAME,
Water Route,536
Northern Route,408
Roundup Route,268
Benge Route,228
Bell Route,199
Overland Water Route,100
Taylor Route,76
Hildebrand Route,70
Bell Route|Whiteley-Drane Route|Overland Water Route,47


,segments
DATAACCESS,
Unrestricted,1975
Internal NPS Only,31


wrote web/data/removal_routes.topojson  4.02 MB


## 6. Assertions
Counts, unique keys, no null keys; every feature has a source and a year (or null + flag); the files read back and match the inputs within the quantization step.

In [13]:
nl = json.loads((WEB_DIR / 'native_land.topojson').read_text())
rr = json.loads((WEB_DIR / 'removal_routes.topojson').read_text())
rj = json.loads((WEB_DIR / 'reservations.json').read_text())
n_tbl = int((tbl[DATE_COLS].min(axis=1).dt.year <= WINDOW_END).sum())
G = nl['objects']['royce_cessions']['geometries']

# Royce: one feature per dated table row; polygons = the ones with a shape
assert len(G) == n_tbl == len(roy), (len(G), n_tbl)
assert sum(g['type'] is not None for g in G) == roy.has_geometry.sum() == len(set(roy.cessnum) & set(shp.CESSNUM.dropna()))
assert all(g['type'] is None for g in G if g['properties']['flag'] == 'no_geometry')
ids = [g['id'] for g in G]; assert None not in ids and len(set(ids)) == len(ids)
assert all(g['properties']['cession_year'] <= WINDOW_END for g in G)

# every feature in every file: a source and a year, or null year + flag
for name, obj in nl['objects'].items():
    for g in obj['geometries']:
        p = g['properties']; assert p['source'] and p['dataset'], (name, g.get('id'))
        assert p['year'] is not None or p.get('year_flag') or p.get('flag'), (name, g.get('id'))
assert {g['properties']['source'] for g in nl['objects']['state_cessions']['geometries']} == {'not_royce'}
assert all(g['properties']['todo'] and g['type'] is None for g in nl['objects']['pre1784_lines']['geometries'])
for r in rj:
    assert r['id'] and r['source'] and r['dataset']
    assert r['start_year'] is not None or r['start_flag'] in ('no_date', 'unparsed')
    assert r['end_year'] is not None or r['end_flag'] in ('no_date', 'continuing', 'after_1840', 'reduced_only', 'unparsed')
assert len(rj) == len(pd.read_csv(RES_CSV)) and len({r['id'] for r in rj}) == len(rj)
assert sum(r['has_coords'] for r in rj) == res.has_coords.sum()
T = rr['objects']['trail_of_tears']['geometries']
assert len(T) == len(tt) and len({g['id'] for g in T}) == len(T) and None not in [g['id'] for g in T]
assert all(g['properties']['year'] is None and g['properties']['year_flag'] == 'undated' for g in T)

# every source column survives
assert set(tbl.columns.drop('_earliest')) | {'CESSNUMSOR', 'SHAPE_AREA', 'SHAPE_LEN'} <= set(G[0]['properties'])
assert set(r1.columns) <= set(nl['objects']['state_cessions']['geometries'][0]['properties'])
assert set(pd.read_csv(RES_CSV).columns) <= set(rj[0])
assert set(tt.columns.drop('geometry')) <= set(T[0]['properties'])

# geometry round trip: decoded shapes within ~2 quantization steps of the inputs
def max_dev(topo, name, src):
    step = max(topo['transform']['scale']) * 2.01
    dev = [a.hausdorff_distance(b) for (a, _), b in zip(topo_decode(topo, name), src) if b is not None]
    return max(dev), step
d, s = max_dev(nl, 'royce_cessions', list(roy.geometry.where(roy.has_geometry, None))); assert d <= s, (d, s); print(f'royce max deviation {d:.5f} deg (limit {s:.5f})')
d, s = max_dev(nl, 'state_cessions', list(sc.geometry)); assert d <= s
d, s = max_dev(rr, 'trail_of_tears', list(tt.geometry)); assert d <= s, (d, s); print(f'trail max deviation {d:.5f} deg (limit {s:.5f})')
print('ok')

royce max deviation 0.00028 deg (limit 0.00101)
trail max deviation 0.00006 deg (limit 0.00024)
ok


In [14]:
# Summary counts by decade (all three files)
con.register('sc_v', sc.drop(columns='geometry'))
print('native_land.topojson:', {k: len(v['geometries']) for k, v in nl['objects'].items()},
      '| royce polygons', sum(g['type'] is not None for g in G), '| royce rows without polygon', sum(g['type'] is None for g in G))
print('reservations.json:', len(rj), 'reserves,', sum(r['has_coords'] for r in rj), 'with a point')
print('removal_routes.topojson:', len(T), 'segments, all undated')
q("""select decade, sum(royce)::int royce_cessions, sum(state)::int state_cession_points, sum(pre)::int pre1784_placeholders from (
       select (cession_year::int // 10) * 10 as decade, 1 royce, 0 state, 0 pre from roy_v
       union all select (year::int // 10) * 10, 0, 1, 0 from sc_v
       union all select (year // 10) * 10, 0, 0, 1 from (select unnest([1763, 1768]) as year))
     group by 1 order by 1""")

native_land.topojson: {'royce_cessions': 724, 'state_cessions': 5, 'pre1784_lines': 2} | royce polygons 717 | royce rows without polygon 7
reservations.json: 66 reserves, 59 with a point
removal_routes.topojson: 2006 segments, all undated


,decade,royce_cessions,state_cession_points,pre1784_placeholders
0,1760,0,0,2
1,1780,6,5,0
2,1790,36,0,0
3,1800,55,0,0
4,1810,71,0,0
5,1820,47,0,0
6,1830,84,0,0
7,1840,18,0,0
8,1850,171,0,0
9,1860,67,0,0


## Notes for review
- **All dated Royce rows are kept** (changed 2026-11-23 from ≤ 1840). Areas first ceded after 1840 (much of WI, MN, IA and the trans-Mississippi) are drawn as not yet ceded for the whole 1784–1840 slider.
- Royce rows with no polygon in the USFS shapefile are kept with null geometry (`flag = no_geometry`); see §1.
- **USFS dates look one day early** (§1a): consistent with a time-zone shift when the table was built. Kept as stored; a fix (+1 day on the 12:00 UTC dates) needs Lara's decision and a historian check against Royce's schedule.
- Ring winding is as read from the shapefile; the encoder does not re-orient rings.